In [2]:
import glob
import math
import ROOT
import numpy as np
ROOT.EnableImplicitMT()
ROOT.gStyle.SetPalette(ROOT.kRainbow)

%jsroot

In [99]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files = glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS_THREE_PI0_NoSmearing/*.root"
)

all_files += glob.glob("/workspace/root_files/kitt/ALL_PHYS2_THREE_PI0_NoSmearing/*.root")
all_files += glob.glob("/workspace/root_files/kitt/ALL_PHYS3_THREE_PI0_NoSmearing/*.root")
all_files += glob.glob("/workspace/root_files/hal/DATA_THREE_PI0_NoSmearing/*.root")

for file in all_files:
    if '205' in file:
        all_files.remove(file)

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df = ROOT.RDataFrame("h1", all_files)
df = df.Define("Kchrec_5", "Kchrec[5]").Define("Knerec_5", "Knerec[5]").Define("pi01Mass", "pi0NeutralFit1[5]").Filter("mctruth != -1 && mctruth != 0 ")

In [85]:
histos = []

histos.append(df.Histo1D(("h_nclu", ";nclu;Counts", 201, 0, 12), "nclu"))
ROOT.RDF.RunGraphs(histos)

1

In [5]:
canva = ROOT.TCanvas("canva", "", 800, 800)

histos[0].Draw()

canva.Draw()

## Optimization of the cut for the control sample tag $K\to\pi^+\pi^-$

In [100]:
df_CS_tag = {}

df_CS_tag["000"] = df.Filter("has00 && mctruth != 0 && mctruth != -1")
df_CS_tag["non000"] = df.Filter("!has00 && mctruth != 0 && mctruth != -1")

total_counts = {}

total_counts["000"] = df_CS_tag["000"].Count().GetValue()
total_counts["non000"] = df_CS_tag["non000"].Count().GetValue()


### Create the histograms

In [101]:
cut_cond = "true"
optimized_variable = "pi01Mass"
var_title = "#Delta_{6} [cm^{2}]"

In [102]:
basic_hist = [df_CS_tag["000"].Histo1D((f"h_{optimized_variable}", f";{var_title};Counts", 201, 0, 300), optimized_variable)]
basic_hist.append(df_CS_tag["non000"].Histo1D((f"h_{optimized_variable}", f";{var_title};Counts", 201, 0, 300), optimized_variable))

In [103]:
ROOT.RDF.RunGraphs(basic_hist)

1

In [104]:
mean = basic_hist[0].GetMean()
max_bin = basic_hist[0].GetMaximumBin()
max_x = basic_hist[0].GetBinCenter(max_bin)
max_y = basic_hist[0].GetBinContent(max_bin)
rms = basic_hist[0].GetRMS()

print(f"Max bin in: {max_x}")
print(f"RMS: {rms}")

canva = ROOT.TCanvas("canva", "", 800, 800)

line = ROOT.TLine(mean, 0.0, mean, max_y)
basic_hist[0].Draw()
basic_hist[1].Draw("SAME")
line.Draw("SAME")
canva.Draw()

Max bin in: 135.07462686567163
RMS: 30.874979187373924


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


### Optimization loop

In [36]:
cut_values = [rms * float(val) for val in np.arange(0, 5.0, 0.1)]

pm_counts = []
nonpm_counts = []

for val in cut_values:
    condition = f"{optimized_variable} < {val}"

    pm_counts.append(df_CS_tag["000"].Filter(condition).Count())
    nonpm_counts.append(df_CS_tag["non000"].Filter(condition).Count())

In [37]:
all_counts = pm_counts + nonpm_counts

ROOT.RDF.RunGraphs(all_counts)

1

In [38]:
pm_counts = [val.GetValue() for val in pm_counts]
nonpm_counts = [val.GetValue() for val in nonpm_counts]

In [39]:
purity = []
significance = []
efficiency = {"000": [], "non000": []}

for nonpm, pm in zip(nonpm_counts, pm_counts):
    purity.append(pm / (nonpm + pm) if (nonpm + pm) > 0 else 0.99)
    significance.append(pm / np.sqrt(nonpm + pm) if (nonpm + pm) > 0 else 0.0)
    efficiency["000"].append(pm / total_counts["000"]) 
    efficiency["non000"].append(nonpm / total_counts["non000"]) 

In [40]:
# canva.SetRightMargin(0.3)

x_arr = np.array(cut_values, dtype=float)
y_arr = np.array(purity, dtype=float)

effpm_arr = np.array(efficiency["000"], dtype=float)
effnonpm_arr = np.array(efficiency["non000"], dtype=float)

graph = ROOT.TGraph(len(x_arr), x_arr, y_arr)
graph_pm = ROOT.TGraph(len(x_arr), x_arr, effpm_arr)
graph_nonpm = ROOT.TGraph(len(x_arr), x_arr, effnonpm_arr)
graph.SetMaximum(1.2)
graph.SetMinimum(0.0)

graph.GetXaxis().SetTitle("Cut value #Delta_{6} [cm^{2}]")
graph.GetYaxis().SetTitle("Value [-]")
graph.SetLineWidth(2)
graph.SetTitle("")
graph.Draw("AL")

graph_pm.SetLineWidth(2)
graph_pm.SetLineColor(ROOT.kRed)
graph_pm.Draw("LSAME")

graph_nonpm.SetLineWidth(2)
graph_nonpm.SetLineColor(ROOT.kBlue)
graph_nonpm.Draw("LSAME")

line = ROOT.TLine(4.0, 0.0, 4.0, 1.2)
line.SetLineStyle(2)
line.Draw("SAME")

legend = ROOT.TLegend(0.5, 0.2, 0.85, 0.45)
legend.AddEntry(graph, "Purity K#rightarrow3#pi^{0}", "l")
legend.AddEntry(graph_pm, "Efficiency K#rightarrow3#pi^{0}", "l")
legend.AddEntry(graph_nonpm, "Efficiency non K#rightarrow3#pi^{0}", "l")
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.Draw()
canva.Draw()

In [69]:
optimal_purity = graph.Eval(500.0)
optimal_efficiency = graph_pm.Eval(500.0)

print(f"Optimal purity: {optimal_purity*100:.2f}%")
print(f"Optimal efficiency: {optimal_efficiency*100:.2f}%")

Optimal purity: 97.36%
Optimal efficiency: 31.71%


In [45]:
pm_counts_arr = np.array(pm_counts, dtype=float)
nonpm_counts_arr = np.array(nonpm_counts, dtype=float)

graph_pm_counts = ROOT.TGraph(len(x_arr), x_arr, pm_counts_arr)
graph_nonpm_counts = ROOT.TGraph(len(x_arr), x_arr, nonpm_counts_arr)

In [47]:
optimal_counts_pm = graph_pm_counts.Eval(300)
optimal_counts_nonpm = graph_nonpm_counts.Eval(300)

print(f"Optimal counts pm: {optimal_counts_pm:.0f}")
print(f"Optimal counts non pm: {optimal_counts_nonpm:.0f}")

Optimal counts pm: 2119754
Optimal counts non pm: 40375


In [48]:
df_tag = df.Filter(f"has000 && bestErrorSixGamma < 300")

In [49]:
condition_neutral_signal = "Chi2NeutralKinFit < 30.0 && mctruth == 4"

In [50]:
counts = []

counts.append(df_tag.Count())
counts.append(df_tag.Filter(condition_neutral_signal).Count())

ROOT.RDF.RunGraphs(counts)

1

In [51]:
counts = [val.GetValue() for val in counts]

In [52]:
print(int(counts[0]), counts[1])

2115634 48703
